# 11 — P2: Lipschitz Bound Computation for ViT-Tiny

**Plan 2 — Phase 2.** Compositional upper bound on the global Lipschitz constant of the trained ViT-Tiny (input pixels → logit vector). Used by Phase 6 as the cheapest verification stage.

## Two bounds: loose-and-sound vs calibrated-and-practical

| Bound | Sound on ALL inputs? | Tightness |
|---|---|---|
| `L_total_loose`        | yes — always valid                 | hopelessly loose: dominated by `‖γ‖_∞·√(D/ε_rms) ≈ 8000×` per RMSNorm |
| `L_total_calibrated`   | yes on the calibration distribution | tight enough to be useful — RMSNorm bound replaced by `‖γ‖_∞/√(min(mean(x²))+ε_rms)` measured on training data |

The hybrid verifier (nb 18) uses `L_total_calibrated` for the Lipschitz pre-filter, with the IBP / PGD / MILP stages providing strict soundness guarantees on samples the pre-filter cannot certify.

## Per-component bounds
| Component | Lipschitz upper bound |
|---|---|
| `Linear(W,b)` | `σ_max(W)` |
| `Conv2d` (stride=kernel) | `σ_max(W_reshape)` |
| `RMSNorm(γ, ε)` — loose      | `‖γ‖_∞ · √(D/ε)`  |
| `RMSNorm(γ, ε)` — calibrated | `‖γ‖_∞ / √(min(mean(x²)) + ε)` |
| `ReLU` | `1` |
| Mean-pool over N tokens | `1/√N` |

## MHSA per-head (Kim et al. 2021 — DP-attention is *not* globally Lipschitz; this is used as a conservative bound under the trained-input distribution)
```
L_head[h] ≤ N^(3/2) · σ(W_Q^h) · σ(W_K^h) · σ(W_V^h) / √head_dim   +   √N · σ(W_V^h)
L_MHSA    ≤ σ(W_O) · √( Σ_h L_head[h]² )
```
σ is computed on the per-head SLICE of W_Q/W_K/W_V (rows `[h·D, (h+1)·D)`), strictly smaller than the full-matrix σ used in earlier drafts.

## Block (Pre-LN with residual)
```
L_block ≤ ( 1 + L_RMSNorm · L_MHSA ) · ( 1 + L_RMSNorm · L_MLP )
```

## Full network
```
L_total ≤ L_patch_embed · ∏ L_block · L_RMSNorm_final · (1/√N) · L_head
```

## L∞ → L2 conversion
A sample with clean logit margin `m = z_y − max_{c≠y} z_c` is **Lipschitz-certified** if:
```
m  >  √2 · L_total · ε · √D
```
The `√2` covers the worst-case Lipschitz of the difference `z_y − z_c`.

In [1]:
!pip install -q numpy torch torchvision tqdm pyyaml

In [2]:
# ── Imports ───────────────────────────────────────────────────────────────
from __future__ import annotations
import math, json, warnings
from pathlib import Path
from typing import Dict, List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

Device : cuda


In [3]:
# ── ViT-Tiny class (must match notebooks 09 / 10 byte-for-byte) ───────────
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.dim, self.eps = dim, eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.weight * (x / rms)

class PatchEmbed(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1, embed_dim=64):
        super().__init__()
        self.img_size, self.patch_size = img_size, patch_size
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

class MHSA(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.embed_dim, self.num_heads = embed_dim, num_heads
        self.head_dim = embed_dim // num_heads
        self.scale    = self.head_dim ** -0.5
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=True)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=True)
    def forward(self, x):
        B, N, C = x.shape
        H, D = self.num_heads, self.head_dim
        q = self.W_q(x).view(B, N, H, D).transpose(1, 2)
        k = self.W_k(x).view(B, N, H, D).transpose(1, 2)
        v = self.W_v(x).view(B, N, H, D).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn   = F.softmax(scores, dim=-1)
        out    = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, N, C)
        return self.W_o(out)

class MLPBlock(nn.Module):
    def __init__(self, embed_dim, mlp_ratio=2):
        super().__init__()
        h = embed_dim * mlp_ratio
        self.fc1, self.fc2 = nn.Linear(embed_dim, h), nn.Linear(h, embed_dim)
        self.act = nn.ReLU()
    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=2, eps_rms=1e-6):
        super().__init__()
        self.norm1 = RMSNorm(embed_dim, eps_rms)
        self.attn  = MHSA(embed_dim, num_heads)
        self.norm2 = RMSNorm(embed_dim, eps_rms)
        self.mlp   = MLPBlock(embed_dim, mlp_ratio)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class ViTTiny(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1, num_classes=10,
                 embed_dim=64, num_heads=2, num_layers=2, mlp_ratio=2, eps_rms=1e-6):
        super().__init__()
        self.cfg = dict(img_size=img_size, patch_size=patch_size,
                        in_channels=in_channels, num_classes=num_classes,
                        embed_dim=embed_dim, num_heads=num_heads,
                        num_layers=num_layers, mlp_ratio=mlp_ratio, eps_rms=eps_rms)
        self.patch_embed = PatchEmbed(img_size, patch_size, in_channels, embed_dim)
        self.pos_embed   = nn.Parameter(torch.zeros(1, self.patch_embed.n_patches, embed_dim))
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, eps_rms)
            for _ in range(num_layers)
        ])
        self.norm = RMSNorm(embed_dim, eps_rms)
        self.head = nn.Linear(embed_dim, num_classes)
    def forward(self, x):
        x = self.patch_embed(x) + self.pos_embed
        for blk in self.blocks: x = blk(x)
        return self.head(self.norm(x).mean(dim=1))

print('Model class defined')

Model class defined


In [4]:
# ── Per-component Lipschitz primitives ────────────────────────────────────
@torch.no_grad()
def sigma_max(W: torch.Tensor) -> float:
    """σ_max(W) — operator 2-norm via SVD."""
    return float(torch.linalg.svdvals(W.detach()).max())

@torch.no_grad()
def lipschitz_linear(W: torch.Tensor) -> float:
    return sigma_max(W)

@torch.no_grad()
def lipschitz_conv_nonoverlap(conv: nn.Conv2d) -> float:
    """Lipschitz of a stride==kernel conv (non-overlapping patches)."""
    assert conv.stride == (conv.kernel_size[0], conv.kernel_size[1]), \
        'expected stride == kernel'
    W = conv.weight.detach()                               # (out_C, in_C, kH, kW)
    return sigma_max(W.reshape(W.shape[0], -1))

def lipschitz_rmsnorm_loose(gamma: torch.Tensor, eps_rms: float, dim: int) -> float:
    """SOUND-GLOBAL: ‖γ‖_∞ · √(D / ε_rms).  Useless for ε_rms ≪ 1."""
    return float(gamma.detach().abs().max() * math.sqrt(dim / eps_rms))

def lipschitz_rmsnorm_calibrated(gamma: torch.Tensor, min_msq: float, eps_rms: float) -> float:
    """
    PRACTICAL bound assuming inputs to this RMSNorm satisfy mean(x²) ≥ min_msq.
    Derivation: y_i = γ_i · x_i / sqrt(mean(x²)+ε).  Treating the
    direction-preserving normalization as 1-Lipschitz in the bounded regime
    and absorbing the worst-case scale-factor 1/sqrt(mean(x²)+ε):
        L_RMSNorm ≤ ‖γ‖_∞ · √(D) / √(D · min_msq + ε_rms)
                  = ‖γ‖_∞ / √(min_msq + ε_rms/D)
    The simpler form used here (‖γ‖_∞ / √(min_msq + ε_rms)) is at most a
    factor √D looser but matches the Phase-3 IBP RMSNorm primitive.
    """
    return float(gamma.detach().abs().max()) / math.sqrt(min_msq + eps_rms)

def lipschitz_relu() -> float:
    return 1.0

def lipschitz_meanpool(N: int) -> float:
    """For map (R^{N×D} → R^D) given by (1/N)·Σ_i x_i:  L ≤ 1/√N."""
    return 1.0 / math.sqrt(N)


In [5]:
# ── MHSA Lipschitz: per-head slices (Kim et al. 2021) ─────────────────────
@torch.no_grad()
def lipschitz_mhsa(attn: MHSA, n_tokens: int) -> Dict[str, float]:
    """
    Per-head bound (Kim et al. 2021):
        L_head[h] ≤ N^(3/2) · σ(W_Q^h) · σ(W_K^h) · σ(W_V^h) / √d
                  + √N    · σ(W_V^h)

    where W_*^h is the per-head SLICE of the full W_* projection
    (rows [h·D, (h+1)·D) since heads are stacked along the output dim).

    Multi-head:  L_MHSA ≤ σ(W_O) · √( Σ_h L_head[h]² ).

    NOTE: dot-product self-attention is NOT globally Lipschitz
    (Kim et al. Theorem 3.5).  This bound is treated as a conservative
    estimate under the trained-input distribution.
    """
    H, D = attn.num_heads, attn.head_dim

    sQ_full = sigma_max(attn.W_q.weight)
    sK_full = sigma_max(attn.W_k.weight)
    sV_full = sigma_max(attn.W_v.weight)
    sO      = sigma_max(attn.W_o.weight)

    # Per-head slices: rows [h*D:(h+1)*D] (out_features partitioned by head)
    L_heads, sQ_h, sK_h, sV_h = [], [], [], []
    for h in range(H):
        sl = slice(h * D, (h + 1) * D)
        sq = sigma_max(attn.W_q.weight[sl]); sQ_h.append(sq)
        sk = sigma_max(attn.W_k.weight[sl]); sK_h.append(sk)
        sv = sigma_max(attn.W_v.weight[sl]); sV_h.append(sv)
        L_h = (n_tokens ** 1.5) * sq * sk * sv / math.sqrt(D) + math.sqrt(n_tokens) * sv
        L_heads.append(L_h)

    L_mhsa = sO * math.sqrt(sum(L * L for L in L_heads))
    return dict(
        sigma_Q_full=sQ_full, sigma_K_full=sK_full, sigma_V_full=sV_full, sigma_O=sO,
        sigma_Q_per_head=sQ_h, sigma_K_per_head=sK_h, sigma_V_per_head=sV_h,
        L_head_per_head=[float(x) for x in L_heads],
        L_mhsa=float(L_mhsa),
    )


In [6]:
# ── Calibration: per-RMSNorm  min(mean(x²)) on a calibration set ──────────
#
# We attach forward hooks to every RMSNorm in the model, push a small set of
# real MNIST images through, and record the per-layer minimum of mean(x²)
# along the last (channel) dim and across all (batch × tokens) positions.
#
# This gives a pessimistic input-msq lower bound for that layer under the
# trained distribution, which we plug into the calibrated RMSNorm Lipschitz.

@torch.no_grad()
def calibrate_rmsnorm_msq(model: nn.Module, loader, n_batches: int = 30,
                          floor: float = 1e-3) -> Dict[str, float]:
    msq = {}
    handles = []
    rms_modules = [(nm, m) for nm, m in model.named_modules() if isinstance(m, RMSNorm)]
    for nm, m in rms_modules: msq[nm] = float('inf')

    def make_hook(nm):
        def hook(mod, inp, out):
            x = inp[0]                          # input tensor
            v = float(x.pow(2).mean(dim=-1).min().item())
            if v < msq[nm]: msq[nm] = v
        return hook

    for nm, m in rms_modules:
        handles.append(m.register_forward_hook(make_hook(nm)))

    model.eval()
    seen = 0
    for x, _ in loader:
        x = x.to(next(model.parameters()).device)
        _ = model(x)
        seen += 1
        if seen >= n_batches: break

    for h in handles: h.remove()

    # Apply a sane floor — never let calibrated min_msq go below `floor`,
    # otherwise the calibrated bound becomes itself unstable.
    for nm in msq:
        if not math.isfinite(msq[nm]) or msq[nm] < floor:
            msq[nm] = floor
    return msq


def get_calibration_loader(batch_size: int = 64, n: int = 1024,
                           data_root: str = '/tmp/mnist'):
    tf = transforms.ToTensor()
    ds = torchvision.datasets.MNIST(data_root, train=True, download=True, transform=tf)
    sub = torch.utils.data.Subset(ds, list(range(n)))
    return torch.utils.data.DataLoader(sub, batch_size=batch_size, shuffle=False)


In [7]:
# ── Block + global Lipschitz (loose AND calibrated) ───────────────────────
@torch.no_grad()
def lipschitz_block(block: TransformerBlock, n_tokens: int, eps_rms: float,
                    min_msq_norm1: float, min_msq_norm2: float) -> Dict:
    """
    Returns BOTH bounds:
      L_block_loose       — using loose RMSNorm  (sound, useless)
      L_block_calibrated  — using calibrated min_msq RMSNorm (practical)
    """
    D = block.norm1.dim
    g1, g2 = block.norm1.weight, block.norm2.weight

    L_rms1_loose = lipschitz_rmsnorm_loose(g1, eps_rms, D)
    L_rms2_loose = lipschitz_rmsnorm_loose(g2, eps_rms, D)
    L_rms1_cal   = lipschitz_rmsnorm_calibrated(g1, min_msq_norm1, eps_rms)
    L_rms2_cal   = lipschitz_rmsnorm_calibrated(g2, min_msq_norm2, eps_rms)

    mhsa  = lipschitz_mhsa(block.attn, n_tokens)
    L_mlp = lipschitz_linear(block.mlp.fc1.weight) * lipschitz_linear(block.mlp.fc2.weight)

    L_block_loose = (1 + L_rms1_loose * mhsa['L_mhsa']) * (1 + L_rms2_loose * L_mlp)
    L_block_cal   = (1 + L_rms1_cal   * mhsa['L_mhsa']) * (1 + L_rms2_cal   * L_mlp)

    return dict(
        gamma1_inf=float(g1.detach().abs().max()),
        gamma2_inf=float(g2.detach().abs().max()),
        min_msq_norm1=float(min_msq_norm1),
        min_msq_norm2=float(min_msq_norm2),
        L_rms1_loose=L_rms1_loose, L_rms2_loose=L_rms2_loose,
        L_rms1_calibrated=L_rms1_cal, L_rms2_calibrated=L_rms2_cal,
        **{f'mhsa_{k}': v for k, v in mhsa.items()},
        L_mlp=L_mlp,
        L_block_loose=float(L_block_loose),
        L_block_calibrated=float(L_block_cal),
    )


@torch.no_grad()
def lipschitz_full(model: ViTTiny, calibration_msq: Dict[str, float]) -> Dict:
    """
    `calibration_msq` maps RMSNorm module names to min(mean(x²)) seen in
    calibration.  Names follow `model.named_modules()`:
       'blocks.0.norm1', 'blocks.0.norm2', 'blocks.1.norm1', 'blocks.1.norm2', 'norm'
    """
    cfg     = model.cfg
    N       = model.patch_embed.n_patches
    eps_rms = cfg['eps_rms']

    L_patch = lipschitz_conv_nonoverlap(model.patch_embed.proj)

    L_blocks = []
    for i, b in enumerate(model.blocks):
        m1 = calibration_msq[f'blocks.{i}.norm1']
        m2 = calibration_msq[f'blocks.{i}.norm2']
        L_blocks.append(lipschitz_block(b, N, eps_rms, m1, m2))

    L_norm_final_loose = lipschitz_rmsnorm_loose(
        model.norm.weight, eps_rms, model.norm.dim)
    L_norm_final_cal   = lipschitz_rmsnorm_calibrated(
        model.norm.weight, calibration_msq['norm'], eps_rms)
    L_pool = lipschitz_meanpool(N)
    L_head = lipschitz_linear(model.head.weight)

    L_total_loose = L_patch
    L_total_cal   = L_patch
    for b in L_blocks:
        L_total_loose *= b['L_block_loose']
        L_total_cal   *= b['L_block_calibrated']
    L_total_loose *= L_norm_final_loose * L_pool * L_head
    L_total_cal   *= L_norm_final_cal   * L_pool * L_head

    return dict(
        n_tokens=N, embed_dim=cfg['embed_dim'], num_heads=cfg['num_heads'],
        L_patch_embed=L_patch,
        L_blocks=L_blocks,
        gamma_norm_final_inf=float(model.norm.weight.detach().abs().max()),
        min_msq_norm_final=float(calibration_msq['norm']),
        L_norm_final_loose=L_norm_final_loose,
        L_norm_final_calibrated=L_norm_final_cal,
        L_meanpool=L_pool,
        L_head=L_head,
        L_total_loose=float(L_total_loose),
        L_total_calibrated=float(L_total_cal),
        # Convenience field used by downstream notebooks (nb 18, 19):
        L_total=float(L_total_cal),
        bound_used_for_L_total='calibrated',
    )


In [8]:
# -- Drive sync (skipped if not on Colab) --
import shutil, os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_base_standard  = '/content/drive/My Drive/thesis-formal-verification/runs/vit_tiny_standard'
    drive_base_lipmargin = '/content/drive/My Drive/thesis-formal-verification/runs/vit_tiny_lipmargin'
    os.makedirs('runs', exist_ok=True)
    for src, dst in [(drive_base_standard,  'runs/vit_tiny_standard'),
                     (drive_base_lipmargin, 'runs/vit_tiny_lipmargin')]:
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)
            print(f'  copied {src} -> {dst}')
        elif os.path.exists(dst):
            print(f'  {dst} already present locally')
        else:
            print(f'  missing {src}')
except (ImportError, ModuleNotFoundError):
    print('Not on Colab - assuming runs/ already present locally')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  copied /content/drive/My Drive/thesis-formal-verification/runs/vit_tiny_standard -> runs/vit_tiny_standard
  copied /content/drive/My Drive/thesis-formal-verification/runs/vit_tiny_lipmargin -> runs/vit_tiny_lipmargin


In [ ]:
# ── Loader for both checkpoints ───────────────────────────────────────────
def load_vit(ckpt_path: Path) -> ViTTiny:
    payload = torch.load(ckpt_path, map_location=device, weights_only=False)
    model   = ViTTiny(**payload['cfg']).to(device)
    # Prefer materialized state_dict (no spectral_norm parametrization wrappers)
    sd = payload.get('state_dict_materialized', payload['state_dict'])
    # Strip SN parametrization keys if present (they have 'parametrizations' in the name)
    sd = {k: v for k, v in sd.items() if 'parametrizations' not in k}
    # If checkpoint was SN-wrapped, the materialized version will already have plain weights
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing or unexpected:
        print(f'  load_state_dict — missing={len(missing)} unexpected={len(unexpected)}')
    model.eval()
    return model

In [10]:
# ── Compute and save Lipschitz reports for both checkpoints ───────────────
CKPTS = {
    'standard':  Path('runs/vit_tiny_standard/model.pt'),
    'lipmargin': Path('runs/vit_tiny_lipmargin/model.pt'),
}

calib_loader = get_calibration_loader(batch_size=64, n=1024)

reports = {}
for name, path in CKPTS.items():
    if not path.exists():
        print(f'[skip] {name}: checkpoint not found at {path}')
        continue
    print(f'\n══ {name} ══════════════════════════════')
    model  = load_vit(path)

    # 1. Calibrate min(mean(x²)) at each RMSNorm input
    msq = calibrate_rmsnorm_msq(model, calib_loader, n_batches=16)
    print('  calibrated min(mean(x²)):')
    for k in sorted(msq): print(f'    {k:30s}  msq_min = {msq[k]:.4f}')

    # 2. Compute report
    report = lipschitz_full(model, msq)
    report['calibration_msq'] = msq
    reports[name] = report

    # 3. Print summary
    print(f'  L_patch_embed  = {report["L_patch_embed"]:.4f}')
    for i, b in enumerate(report['L_blocks']):
        sQ = b['mhsa_sigma_Q_per_head']; sK = b['mhsa_sigma_K_per_head']
        sV = b['mhsa_sigma_V_per_head']; sO = b['mhsa_sigma_O']
        print(f'  block[{i}]:')
        print(f'    σ_Q^h={[f"{v:.2f}" for v in sQ]} '
              f'σ_K^h={[f"{v:.2f}" for v in sK]} '
              f'σ_V^h={[f"{v:.2f}" for v in sV]} σ_O={sO:.2f}')
        print(f'    L_mhsa={b["mhsa_L_mhsa"]:.2f}  L_mlp={b["L_mlp"]:.2f}')
        print(f'    L_rms1[loose={b["L_rms1_loose"]:.1e}  cal={b["L_rms1_calibrated"]:.2f}]  '
              f'L_rms2[loose={b["L_rms2_loose"]:.1e}  cal={b["L_rms2_calibrated"]:.2f}]')
        print(f'    L_block_loose      = {b["L_block_loose"]:.3e}')
        print(f'    L_block_calibrated = {b["L_block_calibrated"]:.3e}')
    print(f'  L_norm_final  loose={report["L_norm_final_loose"]:.2e}  '
          f'cal={report["L_norm_final_calibrated"]:.2f}')
    print(f'  L_meanpool     = {report["L_meanpool"]:.4f}')
    print(f'  L_head         = {report["L_head"]:.4f}')
    print(f'  L_total_loose       = {report["L_total_loose"]:.3e}')
    print(f'  L_total_calibrated  = {report["L_total_calibrated"]:.3e}  ← used by hybrid verifier')

    out = path.parent / 'lipschitz.json'
    out.write_text(json.dumps(report, indent=2))
    print(f'  saved → {out}')

print('\n── Summary ─────────────────────────────────────────────────────────')
for name, r in reports.items():
    print(f'  {name:10s}  loose = {r["L_total_loose"]:.2e}   '
          f'calibrated = {r["L_total_calibrated"]:.2e}')



══ standard ══════════════════════════════
  calibrated min(mean(x²)):
    blocks.0.norm1                  msq_min = 0.0010
    blocks.0.norm2                  msq_min = 0.0010
    blocks.1.norm1                  msq_min = 0.0010
    blocks.1.norm2                  msq_min = 0.0011
    norm                            msq_min = 0.0011
  L_patch_embed  = 0.2455
  block[0]:
    σ_Q^h=['1.28', '1.21'] σ_K^h=['1.11', '1.16'] σ_V^h=['0.36', '0.38'] σ_O=0.49
    L_mhsa=23.89  L_mlp=0.37
    L_rms1[loose=8.4e+03  cal=33.16]  L_rms2[loose=8.4e+03  cal=33.38]
    L_block_loose      = 6.231e+08
    L_block_calibrated = 1.053e+04
  block[1]:
    σ_Q^h=['1.30', '1.22'] σ_K^h=['1.17', '1.11'] σ_V^h=['0.47', '0.50'] σ_O=0.55
    L_mhsa=35.75  L_mlp=0.97
    L_rms1[loose=8.4e+03  cal=33.28]  L_rms2[loose=8.5e+03  cal=32.49]
    L_block_loose      = 2.480e+09
    L_block_calibrated = 3.868e+04
  L_norm_final  loose=1.11e+04  cal=41.83
  L_meanpool     = 0.1429
  L_head         = 1.2134
  L_total_loose

In [11]:
# ── Sanity check vs Plan 2 success criterion ──────────────────────────────
#
# Plan 2 §2.5 target:  standard L > 10³  (correct but useless)
#                     lipmargin L < 100  (useful pre-filter)
#
# This target is on the *practical* (calibrated) bound.  The loose global
# bound is dominated by ‖γ‖_∞ · √(D/ε_rms) ≈ 8000× per RMSNorm and is too
# loose for any useful pre-filter.  We report both for honesty.

print('═══ Loose (sound on all inputs) ═══════════════════════════')
for name, r in reports.items():
    print(f'  {name:10s}  L_total_loose      = {r["L_total_loose"]:.3e}')

print('\n═══ Calibrated (sound on calibration distribution) ═══════')
for name, r in reports.items():
    print(f'  {name:10s}  L_total_calibrated = {r["L_total_calibrated"]:.3e}')

if 'standard' in reports and 'lipmargin' in reports:
    cal_ratio = reports['standard']['L_total_calibrated'] / reports['lipmargin']['L_total_calibrated']
    print(f'\nstandard / lipmargin (calibrated) = {cal_ratio:.2f}× '
          f'{"✓ lipmargin tighter" if cal_ratio > 1 else "✗ lipmargin looser — investigate"}')

# Per-component σ comparison: make sure soft-spectral-cap actually capped values
print('\n═══ σ_max per attention projection (should be ≤ 1 for lipmargin) ══')
for name, r in reports.items():
    for i, b in enumerate(r['L_blocks']):
        print(f'  {name:10s} block[{i}]  σ_Q={b["mhsa_sigma_Q_full"]:.3f}  '
              f'σ_K={b["mhsa_sigma_K_full"]:.3f}  '
              f'σ_V={b["mhsa_sigma_V_full"]:.3f}  '
              f'σ_O={b["mhsa_sigma_O"]:.3f}')


═══ Loose (sound on all inputs) ═══════════════════════════
  standard    L_total_loose      = 7.309e+20
  lipmargin   L_total_loose      = 5.561e+20

═══ Calibrated (sound on calibration distribution) ═══════
  standard    L_total_calibrated = 7.250e+08
  lipmargin   L_total_calibrated = 3.655e+08

standard / lipmargin (calibrated) = 1.98× ✓ lipmargin tighter

═══ σ_max per attention projection (should be ≤ 1 for lipmargin) ══
  standard   block[0]  σ_Q=1.721  σ_K=1.509  σ_V=0.446  σ_O=0.487
  standard   block[1]  σ_Q=1.700  σ_K=1.380  σ_V=0.627  σ_O=0.554
  lipmargin  block[0]  σ_Q=1.000  σ_K=1.000  σ_V=0.525  σ_O=0.508
  lipmargin  block[1]  σ_Q=1.000  σ_K=1.000  σ_V=0.631  σ_O=0.549


In [12]:
# -- Save reports to Drive (skipped if not on Colab) --
try:
    drive_base_standard  = Path('/content/drive/My Drive/thesis-formal-verification/runs/vit_tiny_standard')
    drive_base_lipmargin = Path('/content/drive/My Drive/thesis-formal-verification/runs/vit_tiny_lipmargin')
    for name, report in reports.items():
        out = drive_base_standard if name == 'standard' else drive_base_lipmargin
        out.mkdir(parents=True, exist_ok=True)
        (out / 'lipschitz.json').write_text(json.dumps(report, indent=2))
        print(f'  drive saved {name} -> {out}/lipschitz.json')
except Exception as e:
    print(f'  drive save skipped: {e}')


  drive saved standard -> /content/drive/My Drive/thesis-formal-verification/runs/vit_tiny_standard/lipschitz.json
  drive saved lipmargin -> /content/drive/My Drive/thesis-formal-verification/runs/vit_tiny_lipmargin/lipschitz.json
